## Tahap 0 — Persiapan Lingkungan

**Package:** `os`, `sys`, `json`, `platform`, `random`, `warnings`, `time`, `math`, `subprocess`, `importlib`, `pathlib`, `datetime` (bawaan Python); `numpy`, `pandas`, `torch`, `transformers`, `scikit-learn`, `scipy`, `matplotlib`, `openpyxl`, `pyarrow`; `google.colab.drive`.

Jalankan sel ini **paling awal** setiap kali *runtime* dimulai atau di-*restart*. Sel ini membuat `PATH`, `CFG`, `TARGET`, dan fungsi bantu yang dipakai semua tahap.

In [ ]:
# ══ Tahap 0 — Persiapan Lingkungan ══
# Package : os, sys, json, platform, random, warnings, time, math, subprocess,
#           importlib, pathlib, datetime (bawaan Python); numpy, pandas, torch, transformers,
#           scikit-learn, scipy, matplotlib, openpyxl, pyarrow; google.colab.drive
import os, sys, json, platform, random, warnings, time, math, subprocess
import importlib.util
from importlib.metadata import version as _versi_pkg
from pathlib import Path
from datetime import datetime

t0 = time.perf_counter()

# 1) Determinisme dan senyap: diatur SEBELUM torch/transformers di-import
_cublas_lama = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
_cuda_sudah_aktif = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",        # determinisme cuBLAS (Bagian 3.4)
    "TOKENIZERS_PARALLELISM": "false",           # senyap: peringatan paralelisme tokenizer
    "TRANSFORMERS_VERBOSITY": "error",           # senyap: log transformers
    "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",         # senyap: progress bar unduhan Hugging Face Hub
    "HF_DATASETS_DISABLE_PROGRESS_BARS": "1",    # senyap: progress bar datasets
    "TQDM_DISABLE": "1",                         # senyap: semua progress bar tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",  # jangan muat TensorFlow beserta log-nya
})
warnings.filterwarnings("ignore")

# 2) Pasang hanya package yang belum ada
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}          # nama modul -> nama pip
_kurang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _kurang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_kurang],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, transformers
transformers.logging.set_verbosity_error()       # sembunyikan "Some weights ... newly initialized"
transformers.logging.disable_progress_bar()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)         # Keputusan no. 10 (Bagian 9)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight"})

# 3) Google Drive dan struktur folder (Bagian 3.1)
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount Google Drive gagal. Pilih akun yang menyimpan folder skripsi, "
                           "centang SEMUA izin, lalu jalankan ulang sel ini.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v2")   # folder baru; skripsi_wondr_final tidak disentuh
if not AKAR.parent.is_dir():
    raise RuntimeError("Google Drive belum ter-mount. Jalankan ulang sel ini.")
PATH = {"akar": AKAR, "data": AKAR / "data", "lib": AKAR / "lib", "hasil": AKAR / "hasil",
        "tabel": AKAR / "bab4" / "tabel", "gambar": AKAR / "bab4" / "gambar", "log": AKAR / "log"}
for _p in PATH.values():
    _p.mkdir(parents=True, exist_ok=True)
PATH["xlsx"] = PATH["data"] / "data_wonder_bybni_3.xlsx"
(PATH["lib"] / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." pasti jalan
os.chdir(AKAR)                                    # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 4) Konfigurasi tetap (Bagian 4 dan Bagian 9) -- jangan diubah
CFG = {
    "checkpoint": "indobenchmark/indobert-base-p1",   # Rujukan: Wilie dkk. (2020)
    "ambang_rating": 4,                               # Keputusan: top-2-box, rating >= 4 -> 1 (Morgan & Rego, 2006)
    "proporsi": {"latih": 0.70, "validasi": 0.15, "uji": 0.15},   # Keputusan
    "seed_bagi": 42,                                  # Keputusan
    "kelompok_panjang": ["1-2", "3-10", ">10"],       # Keputusan (bukti Tahap 2)
    "q": 100, "b": 32,                                # Keputusan (Jalur 2): L_max = b*ceil(P_q/b), data latih saja
    "B": 16,                                          # Rujukan: Devlin dkk. (2019)
    "E": 4,                                           # Rujukan: Sun dkk. (2019)
    "patience": 2,                                    # Keputusan
    "clip_norm": 1.0,                                 # Keputusan
    "adam_betas": (0.9, 0.999), "adam_eps": 1e-8,     # Bawaan PyTorch
    "tanpa_wd": ("bias", "LayerNorm.weight"),         # Keputusan no. 1: mengikuti BERT asli
    "default": {"lr": 2e-5, "wr": 0.1, "wd": 0.01},   # Rujukan: Devlin dkk. (2019); Sun dkk. (2019)
    "ruang": {"lr": (1e-5, 5e-5), "wr": (0.0, 0.2), "wd": (0.0, 0.3)},   # Rujukan: Liu & Wang (2021); Bischl dkk. (2023)
    # dekode f in [0,1]^3: lr = 1e-5 * 5**f1 (skala log), wr = 0.2*f2, wd = 0.3*f3   (Hitungan)
    "seed_fitness": (42, 43),                         # Keputusan: fitness = rata-rata 2 seed
    "n_kandidat_rs": 50, "seed_rs": 2024,             # Keputusan
    "pso": {"s": 10, "G_max": 4,                      # Keputusan: 10 x (1 + 4) = 50 evaluasi
            "omega": 0.729, "phi_p": 1.49445, "phi_g": 1.49445,   # Rujukan: Eberhart & Shi (2000); Clerc & Kennedy (2002)
            "V_max": 1.0,                             # Rujukan: Eberhart & Shi (2000)
            "seed": 7},                               # Keputusan
    "M_ragam": 100, "J": 2,                           # Hitungan: 50 RS + 50 PSO; 2 seed per konfigurasi
    "seed_akhir": (42, 43, 44),                       # Keputusan: 3 skema x 3 seed
    "R": 10_000, "seed_bootstrap": 42,                # Rujukan: Efron & Tibshirani (1993)
    "alpha": 0.05, "m_uji": 3,                        # Rujukan: Dror dkk. (2017; 2018); alpha' = 0,05/3
    "daya": 0.80, "delta_target": 0.005,              # Rujukan: Bloom (1995)
    "kembaran_dekat": (339, 595, 2854, 3946, 4128),   # Data (temuan Tahap 7)
}

# 5) Angka target (Bagian 5 dan Bagian 9); Tahap 1-8 sudah diverifikasi ulang 56/56
TARGET = {
    1: {"baris": 7204, "kolom": ["nama", "rating", "ulasan", "tanggal", "likes", "balasan_dev", "versi"],
        "tanggal": ("2026-07-19", "2026-09-11"), "versi": ["1.6.0"],
        "per_bulan": {"2026-07": 1747, "2026-08": 3899, "2026-09": 1558}},
    2: {"baris": 7204, "rating": (1, 5), "kosong": {"balasan_dev": 4834},
        "dibalas_pct": {"1-4": 95.0, "5": 1.0}, "likes_nol_pct": 95.1, "spearman": -0.2483,
        "r3_panjang": (237, 366), "median_kata": [15, 15, 11, 3, 2], "emoji_simbol": 404,
        "huruf_ulang": 227, "huruf_ulang_pct": 3.2, "baca_ulang": 634, "duplikat_pct": 34.8,
        "label_ganda": (33, 1968), "nama_unik": 7153},
    3: {"baris": 7204, "kolom": ["review_id", "ulasan", "rating", "tanggal", "versi"]},
    4: {"per_rating": {1: 1305, 2: 283, 3: 366, 4: 492, 5: 4758}, "positif": 5250,
        "non_positif": 1954, "positif_pct": 72.88, "non_positif_pct": 27.12},
    5: {"berubah": [55, 404, 1518, 227, 634, 403], "kosong": 63, "kosong_rating": {5: 57, 4: 6},
        "hasil": 7141},
    6: {"kunci_kosong": 0, "kunci": 4372, "kunci_seri": 8, "ulasan_seri": 16, "dokumen": 4364,
        "dibuang": 2777, "rincian": (2688, 73, 16), "positif_pct": (72.6, 57.3)},
    7: {"grup": 4293, "latih": (3051, 1303, 1748), "validasi": (657, 282, 375),
        "uji": (656, 280, 376), "bocor": 0, "kembaran_dekat_di_uji": 5},
    8: {"non_positif_pct": {"latih": 42.7, "validasi": 42.9, "uji": 42.7},
        "panjang_pct": {"latih": (14.0, 47.1, 38.9), "validasi": (14.3, 46.9, 38.8),
                        "uji": (14.0, 47.1, 38.9)},
        "p_label": 0.9945, "p_panjang": 0.9998,
        "acuan_validasi": (0.5708, 0.3634), "acuan_uji": (0.5732, 0.3643)},
    9: {"L_max": 128, "terpotong": {"latih": 0, "validasi": 0, "uji": 1}, "terpotong_id": [2741]},
    10: {"T": 764, "T_w": 76, "f1_di_atas": 0.3634, "ulang_seed42_toleransi": 0.0},
    11: {"kandidat": 50, "pelatihan": 100},
    12: {"evaluasi": 50, "angka_acak": 300},
    13: {"M": 100},
    14: {"pelatihan": 9, "determinisme_toleransi": 1e-4},
    15: {"n_uji": 656, "acuan": 0.3643},
    16: {"z1": 2.3940, "z2": 0.8416, "pengganda": 3.2356, "persentil": (0.8333, 99.1667),
         "verifikasi_toleransi": 1e-12},
}

# 6) Fungsi bantu
_STATUS = {"cek_mulai": False}
_ANGKA = (int, float, np.integer, np.floating)

def _angka(x):
    return isinstance(x, _ANGKA) and not isinstance(x, (bool, np.bool_))

def fmt_id(x, d=None):
    """Format angka Indonesia: 7204 -> '7.204'; 0.836 -> '0,8360'. Bukan angka -> str(x)."""
    if not _angka(x):
        return str(x)
    if d is None:
        d = 0 if isinstance(x, (int, np.integer)) else 4
    return f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")

def _tampil(v, d=None):
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, (list, tuple)) and v and all(_angka(x) for x in v):
        return " · ".join(fmt_id(x, d) for x in v)
    if isinstance(v, dict):
        return ", ".join(f"{k}: {_tampil(x, d)}" for k, x in v.items())
    return fmt_id(v, d)

def garis():
    print("─" * 46)

def judul(no, nama, masukan=None, keluaran=None):
    """Kepala blok ringkasan; masukan/keluaran boleh str atau list of str."""
    _STATUS["cek_mulai"] = False
    print(f"══════════ TAHAP {no} — {nama.upper()} ══════════")
    for label, isi in (("Masukan ", masukan), ("Keluaran", keluaran)):
        if isi:
            isi = [isi] if isinstance(isi, str) else list(isi)
            print(f"{label}: {isi[0]}")
            for x in isi[1:]:
                print(" " * 10 + x)
    garis()

def cek(nama, aktual, target, toleransi=0, cara="=", d=None):
    """Cetak ✅/⚠️ tanpa menghentikan program. cara: '=', '>', '>=', '<', '<=', 'memuat'."""
    if not _STATUS["cek_mulai"]:
        garis()
        print("Cek otomatis:")
        _STATUS["cek_mulai"] = True
    if _angka(aktual) and _angka(target):
        if d is None and toleransi > 0:
            d = max(0, round(-math.log10(2 * toleransi)))
        a, t = float(aktual), float(target)
        ok = {"=": abs(a - t) <= toleransi, ">": a > t, ">=": a >= t, "<": a < t, "<=": a <= t}[cara]
    elif cara == "memuat":
        ok = str(target) in str(aktual)
    else:
        try:
            ok = bool(aktual == target)
        except ValueError:                       # array numpy
            ok = bool(np.array_equal(aktual, target))
    tanda = "" if cara in ("=", "memuat") else cara + " "
    print(f"  {'✅' if ok else '⚠️'} {nama} = {_tampil(aktual, d)} (target {tanda}{_tampil(target, d)})")
    return bool(ok)

def selesai(t0):
    dt = time.perf_counter() - t0
    teks = (f"{fmt_id(dt, 1)} detik" if dt < 120 else
            f"{fmt_id(dt / 60, 1)} menit" if dt < 7200 else f"{fmt_id(dt / 3600, 2)} jam")
    print(f"Waktu: {teks}")

def rel(p):
    """Path relatif terhadap folder kerja, untuk ditampilkan."""
    try:
        return str(Path(p).relative_to(PATH["akar"]))
    except ValueError:
        return str(p)

def simpan_tabel(df, nama, index=False):
    """CSV untuk Bab IV (angka mentah) -> bab4/tabel/<nama>.csv"""
    p = PATH["tabel"] / f"{nama}.csv"
    df.to_csv(p, index=index)
    return p

def simpan_gambar(fig, nama):
    """PNG 300 dpi -> bab4/gambar/<nama>.png, lalu ditampilkan satu kali."""
    p = PATH["gambar"] / f"{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return p

def _ke_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, (np.ndarray, Path)):
        return o.tolist() if isinstance(o, np.ndarray) else str(o)
    raise TypeError(f"tidak bisa disimpan ke JSON: {type(o)}")

def simpan_json(obj, path):
    """Tulis JSON secara atomik (berkas .tmp lalu diganti), aman bila Colab terputus."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=_ke_json)
    os.replace(tmp, path)
    return path

def baca_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def atur_seed(s):
    """Satu seed untuk semua sumber keacakan pelatihan (Bagian 3.4)."""
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    transformers.set_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)     # Keputusan no. 10

# 7) GPU dan versi package -> log/versi_package.json
def _versi(pip):
    try:
        return _versi_pkg(pip)
    except Exception:
        return "tidak ditemukan"

_ada_gpu = torch.cuda.is_available()
_gpu = torch.cuda.get_device_name(0) if _ada_gpu else "tidak ada"
_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if _ada_gpu else 0.0
_sesi = {
    "waktu": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "gpu": _gpu, "gpu_memori_gb": round(_mem, 2),
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version() if _ada_gpu else None,
    "package": {pip: _versi(pip) for pip in [*PAKET.values(), "tokenizers"]},
}
_berkas_log = PATH["log"] / "versi_package.json"
_pertama = baca_json(_berkas_log)["pertama"] if _berkas_log.exists() else _sesi
simpan_json({"pertama": _pertama, "terbaru": _sesi}, _berkas_log)
_berubah = [k for k in ("torch", "transformers", "tokenizers") if _pertama["package"].get(k) != _sesi["package"].get(k)]
if _pertama["gpu"] != _sesi["gpu"]:
    _berubah.append("gpu")

# 8) Ringkasan
_ada_data = PATH["xlsx"].exists()
judul(0, "Persiapan Lingkungan",
      masukan=f"{rel(PATH['xlsx'])} ({fmt_id(PATH['xlsx'].stat().st_size / 1024, 1)} KB)" if _ada_data
              else f"{rel(PATH['xlsx'])} (BELUM ADA)",
      keluaran=rel(_berkas_log))
print(f"Folder kerja : {AKAR}")
print(f"Python       : {_sesi['python']}")
print(f"GPU          : {_gpu} ({fmt_id(_mem, 1)} GB) · CUDA {_sesi['cuda']} · cuDNN {_sesi['cudnn']}")
if _kurang:
    print(f"Dipasang     : {', '.join(_kurang)}")
print()
print(f"{'Package':<14}Versi")
for _k, _v in _sesi["package"].items():
    print(f"{_k:<14}{_v}")
cek("berkas data ada", "ya" if _ada_data else "tidak", "ya")
cek("GPU", _gpu, "T4", cara="memuat")
cek("package tersedia", sum(_versi(p) != "tidak ditemukan" for p in PAKET.values()), len(PAKET))
cek("CUBLAS_WORKSPACE_CONFIG sebelum CUDA aktif",
    "tidak (restart runtime)" if (_cuda_sudah_aktif and _cublas_lama != ":4096:8") else "ya", "ya")
cek("mode deterministik PyTorch", "aktif" if torch.are_deterministic_algorithms_enabled() else "mati", "aktif")
cek("versi torch/transformers/tokenizers/GPU sama dengan sesi pertama",
    "sama" if not _berubah else "berubah: " + ", ".join(_berubah), "sama")
selesai(t0)

## Tahap 1 — Pengumpulan Data

**Package:** `pandas` (baca .xlsx, olah tabel), `openpyxl` (mesin baca .xlsx untuk pandas), `pyarrow` (mesin simpan .parquet untuk pandas), `time` (bawaan Python).

Membaca data mentah, membuang kolom indeks `Unnamed*`, lalu menyimpannya ke `hasil/tahap01_mentah.parquet` tanpa mengubah isi. `review_id` dibuat di Tahap 3 (Keputusan no. 6, Bagian 9 `CLAUDE.md`).

In [ ]:
# ══ Tahap 1 — Pengumpulan Data ══
# Package : pandas (baca .xlsx, olah tabel), openpyxl (mesin baca .xlsx untuk pandas),
#           pyarrow (mesin simpan .parquet untuk pandas), time (bawaan Python)
import time
import pandas as pd

t0 = time.perf_counter()
T = TARGET[1]
BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
         "Agustus", "September", "Oktober", "November", "Desember"]

def tgl_id(t):                                   # 2026-07-19 -> "19 Juli 2026"
    return f"{t.day} {BULAN[t.month - 1]} {t.year}"

# 1) Baca data mentah (.xlsx)
mentah = pd.read_excel(PATH["xlsx"], engine="openpyxl")

# 2) Buang kolom indeks bawaan Excel (Unnamed*); isi data tidak diubah, review_id dibuat di Tahap 3
buang = [c for c in mentah.columns if str(c).startswith("Unnamed")]
df = mentah.drop(columns=buang)

# 3) Simpan keluaran, lalu pastikan berkas yang dibaca ulang identik
keluar = PATH["hasil"] / "tahap01_mentah.parquet"
df.to_parquet(keluar, index=False)
identik = df.equals(pd.read_parquet(keluar))

# 4) Ringkasan + cek otomatis
tgl = pd.to_datetime(df["tanggal"])              # hanya untuk ringkasan; kolom asli tidak diubah
per_bulan = tgl.dt.strftime("%Y-%m").value_counts().sort_index()
versi = sorted(df["versi"].astype(str).unique())

judul(1, "Pengumpulan Data",
      masukan=f"{rel(PATH['xlsx'])} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"Kolom dibuang  : {', '.join(map(str, buang)) if buang else '-'}")
print(f"Rentang tanggal: {tgl_id(tgl.min())} – {tgl_id(tgl.max())}")
print(f"Versi aplikasi : {', '.join(versi)}")
print()
print(f"{'Bulan':<16}{'Ulasan':>7}")
for k, v in per_bulan.items():
    print(f"{BULAN[int(k[5:]) - 1] + ' ' + k[:4]:<16}{fmt_id(v):>7}")
print(f"{'Total':<16}{fmt_id(int(per_bulan.sum())):>7}")

cek("jumlah baris", len(df), T["baris"])
cek("nama dan urutan kolom", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(df.columns), "sesuai")
cek("rentang tanggal", f"{tgl.min().date()} s.d. {tgl.max().date()}", f"{T['tanggal'][0]} s.d. {T['tanggal'][1]}")
cek("nilai unik versi", ", ".join(versi), ", ".join(T["versi"]))
cek("ulasan per bulan", per_bulan.to_dict(), T["per_bulan"])
cek("parquet identik dengan isi .xlsx", "ya" if identik else "tidak", "ya")
selesai(t0)

## Tahap 2 — Audit Struktur dan Kualitas Data

**Package:** `pandas` (olah tabel), `scipy.stats.spearmanr` (korelasi *likes*–*rating*), `lib.wondr_teks` (pola teks Bagian 6; memakai `re`, `unicodedata`, `numpy`, `pandas`), `hashlib`, `importlib`, `time` (bawaan Python).

Sel pertama menulis `lib/wondr_teks.py` (kode acuan Bagian 6) karena pola teksnya pertama kali dibutuhkan di tahap ini; Tahap 5–7 cukup meng-*import*-nya. Sel kedua memeriksa kualitas data tanpa mengubahnya.

In [ ]:
%%writefile lib/wondr_teks.py
# lib/wondr_teks.py -- kode acuan Bagian 6 CLAUDE.md, disalin persis.
# Karakter non-ASCII pada pola EMO dan SIMBOL ditulis sebagai escape (Keputusan no. 9);
# pola hasilnya identik dengan kode acuan dan dicek lewat sidik jari di Tahap 2.
import re, unicodedata
import numpy as np, pandas as pd

# ── Pola teks (Tahap 2 dan Tahap 5 memakai pola yang sama) ──
EMO = re.compile("[\U0001F000-\U0001FAFF\U0001F1E6-\U0001F1FF\u2600-\u27bf\u2b00-\u2bff"
                 "\u2300-\u23ff\u2190-\u21ff\u2900-\u297f\u3030\u303d\u3297\u3299"
                 "\u00a9\u00ae\u203c\u2049\u2122\u2139\u24c2\u25aa-\u25fe\ufe0f\u200d"
                 "\U000E0020-\U000E007F]")
SIMBOL = re.compile("[\u2600-\u27bf\u2b00-\u2bff\ufe0f\u200d\U0001F3FB-\U0001F3FF]")
HURUF = re.compile(r"([^\W\d_])\1{2,}")      # huruf yang berulang >= 3 kali (hanya huruf)
BACA = re.compile(r"[!?.,;:]{2,}")           # tanda baca berurutan >= 2

LANGKAH = [  # (nama, fungsi) -- urutan Tahap 5, jangan diubah
    ("1. Normalisasi Unicode NFKC", lambda s: unicodedata.normalize("NFKC", s)),
    ("2. Hapus emoji dan simbol", lambda s: SIMBOL.sub(" ", EMO.sub(" ", s))),
    ("3. Case folding", lambda s: s.lower()),
    ("4. Huruf berulang >=3 -> 2", lambda s: HURUF.sub(r"\1\1", s)),
    ("5. Tanda baca berulang -> 1", lambda s: BACA.sub(lambda m: m.group(0)[0], s)),
    ("6. Rapikan spasi", lambda s: re.sub(r"\s+", " ", s).strip()),
]

def bersihkan(s):
    s = str(s)
    for _, f in LANGKAH:
        s = f(s)
    return s

# ── Tahap 6: deduplikasi ──
def kunci_dedup(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s))   # sisakan a-z, 0-9, spasi
    s = re.sub(r"(.)\1+", r"\1", s)          # karakter berurutan yang sama -> satu
    return re.sub(r"\s+", " ", s).strip()

def deduplikasi(d):
    d = d.copy()
    d["dedup_key"] = d["text_clean"].map(kunci_dedup)
    d = d[d["dedup_key"] != ""]
    suara = (d.groupby(["dedup_key", "label"]).size().unstack(fill_value=0)
               .reindex(columns=[0, 1], fill_value=0))
    suara.columns = ["n_np", "n_p"]
    seri = suara["n_np"] == suara["n_p"]
    suara["mayor"] = (suara["n_p"] > suara["n_np"]).astype(int)
    sah = suara[~seri]
    x = d[d["dedup_key"].isin(sah.index)].copy()
    x = x[x["label"] == x["dedup_key"].map(sah["mayor"])]          # anggota berlabel mayoritas
    frek = x.groupby(["dedup_key", "text_clean"]).size().rename("_f").reset_index()
    x = x.merge(frek, on=["dedup_key", "text_clean"])
    wakil = (x.sort_values(["dedup_key", "_f", "review_id"], ascending=[True, False, True])
               .groupby("dedup_key").head(1)["review_id"])           # teks tersering, lalu id terkecil
    hasil = d[d["review_id"].isin(set(wakil))].copy()
    hasil["label"] = hasil["dedup_key"].map(sah["mayor"]).astype(int)
    return hasil.sort_values("review_id").reset_index(drop=True), suara, seri

# ── Tahap 7: pembagian data berbasis grup ──
KATA_FUNGSI = {"dan", "yang", "nya", "ini", "itu", "ya", "sih", "deh", "juga", "aja", "saja", "nih",
               "kok", "dong", "lah", "di", "ke", "dari", "untuk", "pada", "dengan", "yg", "jg", "utk", "dgn"}

def kunci_grup(s):                      # hanya untuk pengelompokan; teks tidak diubah
    k = set(str(s).split())
    inti = k - KATA_FUNGSI
    return " ".join(sorted(inti)) if inti else " ".join(sorted(k))

def kelompok_panjang(n_kata):
    return "1-2" if n_kata <= 2 else ("3-10" if n_kata <= 10 else ">10")

PROPORSI = {"latih": 0.70, "validasi": 0.15, "uji": 0.15}   # urutan kunci menentukan pemecah seri

def bagi_data(dd, seed=42):
    dd = dd.copy()
    dd["group_key"] = dd["text_clean"].map(kunci_grup)
    dd["n_kata"] = dd["text_clean"].str.split().str.len()
    label_id_kecil = dd.sort_values("review_id").groupby("group_key")["label"].first()
    g = (dd.groupby("group_key")
           .agg(n=("review_id", "size"), n_p=("label", "sum"), kata=("n_kata", "median"))
           .reset_index())
    mayor = np.where(g.n_p * 2 > g.n, 1, np.where(g.n_p * 2 < g.n, 0, -1))
    g["label_grup"] = np.where(mayor == -1, g.group_key.map(label_id_kecil).to_numpy(), mayor).astype(int)
    g["kelompok_panjang"] = g["kata"].map(kelompok_panjang)
    rng = np.random.default_rng(seed)                     # satu generator untuk semua strata
    alokasi = {}
    for _, sub in g.groupby(["label_grup", "kelompok_panjang"], sort=True):
        sub = sub.iloc[rng.permutation(len(sub))]
        total = int(sub.n.sum())
        target = {b: p * total for b, p in PROPORSI.items()}
        terisi = {b: 0 for b in PROPORSI}
        for kunci, n in zip(sub.group_key, sub.n):
            b = min(terisi, key=lambda b: terisi[b] / target[b])   # keterisian terkecil
            alokasi[kunci] = b
            terisi[b] += int(n)
    dd["split"] = dd["group_key"].map(alokasi)
    dd["kelompok_panjang"] = dd["n_kata"].map(kelompok_panjang)
    return dd, g

In [ ]:
# ══ Tahap 2 — Audit Struktur dan Kualitas Data ══
# Package : pandas (olah tabel), scipy.stats.spearmanr (korelasi likes-rating),
#           lib.wondr_teks (pola EMO, SIMBOL, HURUF, BACA), hashlib, importlib, time (bawaan Python)
import time, hashlib, importlib
import pandas as pd
from scipy.stats import spearmanr
import lib.wondr_teks as wt
importlib.reload(wt)                                  # pakai isi lib/wondr_teks.py yang terbaru

t0 = time.perf_counter()
T = TARGET[2]
SIDIK_POLA = "26c43d7121739a69"                       # Hitungan: sidik jari pola Bagian 6 (Keputusan no. 9)

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
df = pd.read_parquet(masuk)

# 2) Pemeriksaan -- data hanya dibaca, tidak diubah
u = df["ulasan"].astype(str)
n_kata = u.str.split().str.len()                      # kata = dipisah spasi
kosong = {k: int(v) for k, v in df.isna().sum().items() if v}
dibalas = df["balasan_dev"].notna()
pct_14 = dibalas[df["rating"] <= 4].mean() * 100
pct_5 = dibalas[df["rating"] == 5].mean() * 100
likes_nol = (df["likes"] == 0).mean() * 100
rho = spearmanr(df["likes"], df["rating"]).statistic
r3 = df["rating"] == 3
r3_panjang = int((r3 & (n_kata >= 8)).sum())
median_kata = [float(n_kata[df["rating"] == r].median()) for r in range(1, 6)]
emoji = int(u.map(lambda s: bool(wt.EMO.search(s) or wt.SIMBOL.search(s))).sum())
huruf = u.str.lower().map(lambda s: bool(wt.HURUF.search(s)))
baca = int(u.map(lambda s: bool(wt.BACA.search(s))).sum())
dup_pct = df["ulasan"].duplicated().mean() * 100
label_audit = (df["rating"] >= CFG["ambang_rating"]).astype(int)   # hanya untuk audit; label resmi di Tahap 4
teks_kecil = u.str.lower().str.strip()
n_label = label_audit.groupby(teks_kecil).nunique()
ganda = n_label[n_label == 2].index
label_ganda = (len(ganda), int(teks_kecil.isin(ganda).sum()))
nama_unik = int(df["nama"].nunique())
rentang = (int(df["rating"].min()), int(df["rating"].max()))
versi = sorted(df["versi"].astype(str).unique())
sidik = hashlib.sha256("\n".join(p.pattern for p in (wt.EMO, wt.SIMBOL, wt.HURUF, wt.BACA))
                       .encode("utf-8")).hexdigest()[:16]

# 3) Tabel temuan -> keputusan, lalu simpan (CSV berisi angka mentah)
f = fmt_id
TEMUAN = [  # (temuan, angka tampil, keputusan yang dibenarkan, tahap pemakai, {metrik: nilai mentah})
    ("Nilai kosong", f"balasan_dev {f(kosong.get('balasan_dev', 0))}; kolom lain 0",
     "balasan_dev dibuang", "3", {"kosong_balasan_dev": kosong.get("balasan_dev", 0),
                                  "kosong_kolom_lain": sum(v for k, v in kosong.items() if k != "balasan_dev")}),
    ("Balasan pengembang", f"dibalas: rating 1–4 {f(pct_14, 1)}% · 5 {f(pct_5, 1)}%",
     "dibuang: membocorkan label", "3", {"dibalas_pct_rating_1_4": pct_14, "dibalas_pct_rating_5": pct_5}),
    ("Likes", f"nol {f(likes_nol, 1)}%; rho Spearman {f(rho, 4)}",
     "dibuang: tidak ada saat prediksi", "3", {"likes_nol_pct": likes_nol, "spearman_likes_rating": rho}),
    ("Nama pengguna", f"{f(nama_unik)} nama unik",
     "dibuang: privasi, bukan fitur", "3", {"nama_unik": nama_unik}),
    ("Versi aplikasi", f"{len(versi)} nilai ({', '.join(versi)})",
     "informasi pendukung saja", "3", {"versi_unik": len(versi)}),
    ("Rentang rating", f"{rentang[0]}–{rentang[1]}",
     "label top-2-box (rating >= 4)", "4", {"rating_min": rentang[0], "rating_max": rentang[1]}),
    ("Rating 3 dengan >= 8 kata", f"{f(r3_panjang)} dari {f(int(r3.sum()))}",
     "rating 3 dipertahankan (Non-Positif)", "4", {"rating3_panjang": r3_panjang, "rating3_total": int(r3.sum())}),
    ("Median kata per rating 1–5", " · ".join(f(m, 0) for m in median_kata),
     "kelompok panjang 1–2, 3–10, >10", "7", {f"median_kata_rating_{r}": m for r, m in zip(range(1, 6), median_kata)}),
    ("Emoji atau simbol", f"{f(emoji)} baris",
     "diganti spasi", "5", {"emoji_simbol": emoji}),
    ("Huruf berulang >= 3", f"{f(int(huruf.sum()))} baris ({f(huruf.mean() * 100, 1)}%)",
     "dipendekkan menjadi 2", "5", {"huruf_ulang": int(huruf.sum()), "huruf_ulang_pct": huruf.mean() * 100}),
    ("Tanda baca berulang", f"{f(baca)} baris",
     "dipendekkan menjadi 1", "5", {"baca_ulang": baca}),
    ("Duplikat persis", f"{f(dup_pct, 1)}% baris",
     "deduplikasi sebelum pembagian", "6", {"duplikat_pct": dup_pct}),
    ("Teks berlabel ganda", f"{f(label_ganda[0])} teks, {f(label_ganda[1])} baris",
     "label mayoritas; seri dibuang", "6", {"label_ganda_teks": label_ganda[0], "label_ganda_baris": label_ganda[1]}),
]
keluar = simpan_tabel(pd.DataFrame([{"temuan": t, "metrik": m, "nilai": v, "keputusan": k, "tahap": th}
                                    for t, _, k, th, mm in TEMUAN for m, v in mm.items()]), "tahap02_kualitas")

# 4) Ringkasan + cek otomatis
judul(2, "Audit Struktur dan Kualitas Data",
      masukan=[f"{rel(masuk)} ({f(len(df))} baris)", "lib/wondr_teks.py (pola teks Bagian 6)"],
      keluaran=f"{rel(keluar)} ({sum(len(x[4]) for x in TEMUAN)} baris)")
print(f"{'Temuan':<28}{'Angka':<38}{'Keputusan yang dibenarkan':<38}Tahap")
for t, a, k, th, _ in TEMUAN:
    print(f"{t:<28}{a:<38}{k:<38}{th}")

cek("jumlah baris tetap", len(df), T["baris"])
cek("sidik jari pola teks", sidik, SIDIK_POLA)
cek("rentang rating", rentang, T["rating"])
cek("nilai kosong", kosong, T["kosong"])
cek("dibalas, rating 1–4 (%)", pct_14, T["dibalas_pct"]["1-4"], toleransi=0.05)
cek("dibalas, rating 5 (%)", pct_5, T["dibalas_pct"]["5"], toleransi=0.05)
cek("likes bernilai nol (%)", likes_nol, T["likes_nol_pct"], toleransi=0.05)
cek("rho Spearman likes-rating", rho, T["spearman"], toleransi=5e-5)
cek("rating 3 >= 8 kata (jumlah, total)", (r3_panjang, int(r3.sum())), T["r3_panjang"])
cek("median kata per rating", median_kata, T["median_kata"], d=0)
cek("baris dengan emoji atau simbol", emoji, T["emoji_simbol"])
cek("baris dengan huruf berulang", int(huruf.sum()), T["huruf_ulang"])
cek("baris dengan huruf berulang (%)", huruf.mean() * 100, T["huruf_ulang_pct"], toleransi=0.05)
cek("baris dengan tanda baca berulang", baca, T["baca_ulang"])
cek("duplikat persis (%)", dup_pct, T["duplikat_pct"], toleransi=0.05)
cek("teks berlabel ganda (teks, baris)", label_ganda, T["label_ganda"])
cek("nama unik", nama_unik, T["nama_unik"])
selesai(t0)